# 1.4 — Write foreground database

Validates the reviewed process structure, inventory and Brightway mappings,
then — only if the plan is ready **and** `CONFIRM_WRITE_FOREGROUND = True` in
[ai_lca_config.py](ai_lca_config.py) — writes a new Brightway foreground
database named `NEW_FOREGROUND_DB_NAME`.

The writer is deliberately strict (it blocks on missing amounts/units,
unmapped flows, unresolved directions, or mismatched units — see the project
README) and **never overwrites an existing database**. Once written, every
process here becomes selectable from the electrolyser LCA side via
`H.list_process_activities("<NEW_FOREGROUND_DB_NAME>")` in `lca_helpers.py`,
or directly on the Setup LCA Streamlit page.

In [ ]:
import pandas as pd

import ai_lca_config as cfg
cfg.print_config()

from ai_lca.brightway_writer import build_write_plan, write_foreground_database
from ai_lca.export import review_bundle_to_json
from ai_lca.notebook_helpers import load_extraction, run_output_dir

run_dir = run_output_dir(cfg.OUTPUT_DIR, cfg.RUN_LABEL)
raw_path = run_dir / "1_extraction_raw.json"
reviewed_path = run_dir / "1_1_extraction_reviewed.json"
inventory_path = run_dir / "1_2_inventory_reviewed.csv"
mapping_path = run_dir / "1_3_mapping.csv"
for p in (raw_path, reviewed_path, inventory_path, mapping_path):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found — run the earlier 1.x notebooks first.")

original_extraction = load_extraction(raw_path)
extraction = load_extraction(reviewed_path)
inventory_df = pd.read_csv(inventory_path)
mapping_df = pd.read_csv(mapping_path)
print(f"Loaded {len(extraction.processes)} process(es), {len(inventory_df)} flow row(s), "
      f"{len(mapping_df)} mapping(s).")

## Validate the write plan

In [ ]:
plan = build_write_plan(extraction, inventory_df, mapping_df)

if plan.ready:
    print(f"READY: {len(extraction.processes)} process(es), {len(plan.exchanges)} quantitative exchange(s).")
else:
    print("NOT READY — resolve these before writing:")
    for b in plan.blockers:
        print("  -", b)
    print()
    print("Most blockers trace back to 1.2.paper_inventory_review.ipynb (missing amount/unit,")
    print("unresolved direction) or 1.3.paper_brightway_matching.ipynb (missing/blocked mapping).")

if plan.warnings:
    print()
    print("Warnings (non-blocking):")
    for w in plan.warnings:
        print("  -", w)

## Write the database

In [ ]:
if not plan.ready:
    print("Skipped: write plan is not ready. See blockers above.")
elif not cfg.CONFIRM_WRITE_FOREGROUND:
    print("Skipped: set CONFIRM_WRITE_FOREGROUND = True in ai_lca_config.py once you've reviewed")
    print("the plan above, then re-run this cell.")
else:
    report = write_foreground_database(
        project_name=cfg.BRIGHTWAY_PROJECT,
        database_name=cfg.NEW_FOREGROUND_DB_NAME,
        extraction=extraction,
        inventory_df=inventory_df,
        mapping_df=mapping_df,
    )
    print(f"Created Brightway database {report['database']!r}:")
    print(f"  {report['processes_created']} process(es), {report['exchanges_created']} exchange(s)")
    if report["brightway_location"]:
        print(f"  Location: {report['brightway_location']}")
    for w in report["warnings"]:
        print("  WARNING:", w)

## Export reproducible review bundle

In [ ]:
bundle = review_bundle_to_json(extraction, inventory_df, mapping_df, original_extraction=original_extraction)
bundle_path = run_dir / "1_4_review_bundle.json"
bundle_path.write_text(bundle)
print("Saved review bundle to:", bundle_path)

## Done

If the database was written, `cfg.NEW_FOREGROUND_DB_NAME` is now a foreground
database in the `hydrogen-smr` Brightway project — pick it up from either
side:

- **Notebooks**: `H.list_process_activities("<NEW_FOREGROUND_DB_NAME>")` in
  `lca_helpers.py`, then pass it through `TECH_SOURCE_OVERRIDES` in
  `dashboard_config.py` the same way `4.1`/`5` already do.
- **Streamlit**: pick it straight from the foreground database dropdown on the
  **Setup LCA** page.